In [ ]:
import pandas as pd
import numpy as np
import warnings
from typing import Any
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.vector_ar.vecm import coint_johansen
from tiportfolio.metrics import compute_metrics
from tiportfolio.allocation import DollarNeutral
from tiportfolio.data import fetch_prices
from tiportfolio.backtest import run_backtest
from tiportfolio.engine import ScheduleBasedEngine
from tiportfolio.calendar import Schedule

class CointegrationDollarNeutral(DollarNeutral):
    """
    A money-neutral pair trading strategy based on co-integration and Z-Score signals.
    """
    def __init__(self, symbol_A: str, symbol_B: str, cash_symbol: str, 
                 hedge_ratio: float, lookback_window: int = 252, 
                 z_entry: float = 2.0, z_exit: float = 0.0, 
                 book_size: float = 0.5, tolerance: float = 0.05):
        
        self.symbol_A = symbol_A
        self.symbol_B = symbol_B
        self.hedge_ratio = hedge_ratio
        self.lookback_window = lookback_window
        self.z_entry = z_entry
        self.z_exit = z_exit
        self.target_book_size = book_size # The proportion of funds to be allocated to the storage target
        
        self.current_signal = 0 # 0: Empty-handed, 1: Long spread (long A, short B), -1: Short spread (short A, long B)
        
        super().__init__(
            long_weights={symbol_A: 1.0},
            short_weights={symbol_B: 1.0},
            cash_symbol=cash_symbol,
            book_size=0.0, 
            tolerance=tolerance
        )

    def get_target_weights(self, date: pd.Timestamp, total_equity: float, 
                           positions_dollars: dict[str, float], 
                           prices_row: pd.Series, **context: Any) -> dict[str, float]:
        """
        Calculate Z-Score -> 
        Determine signal -> 
        Dynamically adjust the long/short target and capital ratio -> 
        Assign final weight to parent category.
        """
        # 1. Retrieve historical prices from the context
        prices_history = context.get("prices_history")
        
        # If historical data is insufficient to calculate moving averages and standard deviations, remain uninvested
        if prices_history is None or len(prices_history) < self.lookback_window:
            self.long_book_size = 0.0
            self.short_book_size = 0.0
            return super().get_target_weights(date, total_equity, positions_dollars, prices_row, **context)

        # 2. Retrieve historical prices from the past lookback_window days to calculate the Z-Score
        hist = prices_history[[self.symbol_A, self.symbol_B]].tail(self.lookback_window)
        
        # Calculate the historical log spread
        log_hist_A = np.log(hist[self.symbol_A])
        log_hist_B = np.log(hist[self.symbol_B])
        historical_spreads = log_hist_A - (self.hedge_ratio * log_hist_B)
        
        # Calculate the mean and standard deviation
        spread_mean = historical_spreads.mean()
        spread_std = historical_spreads.std()
        
        # Calculate today's Z-Score
        if spread_std > 0:
            current_log_A = np.log(prices_row[self.symbol_A])
            current_log_B = np.log(prices_row[self.symbol_B])
            current_spread = current_log_A - (self.hedge_ratio * current_log_B)
            z_score = (current_spread - spread_mean) / spread_std
        else:
            z_score = 0.0

        # 3. Determine the Mean Reversion of Trading Signals
        if z_score > self.z_entry:
            self.current_signal = -1  # Excessive price spread: Short A, Long B
        elif z_score < -self.z_entry:
            self.current_signal = 1   # Price spread too small: Long A, Short B
        elif abs(z_score) < self.z_exit:
            self.current_signal = 0   # Regression to the mean: Close position and go empty

        # 4. 
        if self.current_signal == 1:
            self.long_weights = {self.symbol_A: 1.0}
            self.short_weights = {self.symbol_B: 1.0}
            self.long_book_size = self.target_book_size
            self.short_book_size = self.target_book_size
            
        elif self.current_signal == -1:
            self.long_weights = {self.symbol_B: 1.0}
            self.short_weights = {self.symbol_A: 1.0}
            self.long_book_size = self.target_book_size
            self.short_book_size = self.target_book_size
            
        else:
            # The parent category will automatically allocate all funds to cash_symbol
            self.long_book_size = 0.0
            self.short_book_size = 0.0

        return super().get_target_weights(date, total_equity, positions_dollars, prices_row, **context)

print("✅ CointegrationDollarNeutral class defined successfully!")

✅ CointegrationDollarNeutral class defined successfully!


In [24]:
def test_johansen_cointegration(log_prices_data, det_order=0, k_ar_diff=1):
    """
    Perform Johansen cointegration test on log price data.
    
    Parameters:
    -----------
    log_prices_data : pd.DataFrame
        DataFrame with log prices for multiple assets
    det_order : int, default 0
        Deterministic order (0: no constant, 1: constant, 2: constant + trend)
    k_ar_diff : int, default 1
        Number of lagged differences in the model
    
    Returns:
    --------
    dict
        Dictionary containing test results:
        - 'eigenvectors': Eigenvectors from Johansen test
        - 'eigenvalues': Eigenvalues from Johansen test
        - 'trace_stat': Trace statistics
        - 'max_eig_stat': Maximum eigenvalue statistics
        - 'trace_crit': Trace critical values
        - 'max_eig_crit': Max eigenvalue critical values
        - 'coint_rank': Number of cointegrating relationships
    """
    try:
        # Perform Johansen cointegration test
        result = coint_johansen(log_prices_data, det_order, k_ar_diff)
        
        # Extract results
        eigenvectors = result.evec
        eigenvalues = result.eig
        trace_stat = result.lr1
        max_eig_stat = result.lr2
        trace_crit = result.cvt
        max_eig_crit = result.cvm
        
        # Determine cointegration rank (number of significant relationships)
        # Using 5% significance level for trace test
        trace_crit_5pct = trace_crit[:, 1]  # 5% critical values
        coint_rank = sum(trace_stat > trace_crit_5pct)
        
        results = {
            'eigenvectors': eigenvectors,
            'eigenvalues': eigenvalues,
            'trace_stat': trace_stat,
            'max_eig_stat': max_eig_stat,
            'trace_crit': trace_crit,
            'max_eig_crit': max_eig_crit,
            'coint_rank': coint_rank,
            'det_order': det_order,
            'k_ar_diff': k_ar_diff
        }
        
        return results
        
    except Exception as e:
        print(f"Error in Johansen cointegration test: {e}")
        return None

In [ ]:
# Define pairings and cash
symbols = ['KO', 'PEP']
cash_symbol = 'BIL' 
all_symbols = symbols + [cash_symbol]

start_date = '2018-01-01'
end_date = '2024-12-31'

SYMBOL1 = symbols[0]
SYMBOL2 = symbols[1]
TOTAL_CAPITAL = 100000

print(f"Loading data for {all_symbols} from {start_date} to {end_date}...")

try:
    prices_data = fetch_prices(all_symbols, start_date, end_date)
    print(f"✅ Successfully fetched data for {len(prices_data)} symbols")
except Exception as e:
    print(f"❌ Error fetching data: {e}")
    print("Falling back to local test data...")
    # Fallback to local test data if available
    import os
    # Use symbolic variables for fallback filenames (you may need to adjust these based on your actual data files)
    fallback_file1 = f'tests/data/{SYMBOL1.lower()}_2018_2024.csv'
    fallback_file2 = f'tests/data/{SYMBOL2.lower()}_2018_2024.csv'
    
    if os.path.exists(fallback_file1) and os.path.exists(fallback_file2):
        symbol1_data = pd.read_csv(fallback_file1, index_col=0, parse_dates=True)
        symbol2_data = pd.read_csv(fallback_file2, index_col=0, parse_dates=True)
        prices_data = {SYMBOL1: symbol1_data, SYMBOL2: symbol2_data}
        print("Using local test data")
    else:
        raise FileNotFoundError("No data available")
    raise

# The closing price is used to calculate the Hedge Ratio.
close_prices = pd.DataFrame()
for sym in symbols:
    close_prices[sym] = prices_data[sym]['close']
close_prices = close_prices.dropna()

# Calculate Johansen Cointegration to find the optimal Hedge Ratio
print("\nCalculating Johansen Cointegration for KO/PEP...")
log_prices = np.log(close_prices)

# 使用前面寫好的函數計算 (假設 det_order=1, k_ar_diff=1 表現較好)
coint_results = test_johansen_cointegration(log_prices, det_order=1, k_ar_diff=1)

if coint_results and coint_results['coint_rank'] > 0:
    hedge_ratio_info = calculate_hedge_ratio(coint_results)
    selected_hedge_ratio = hedge_ratio_info['hedge_ratio'] if hedge_ratio_info else 1.0
    print(f"✅ Cointegration found! Using Hedge Ratio: {selected_hedge_ratio:.4f}")
else:
    print("⚠️ No strong cointegration found. Using default Hedge Ratio: 1.0")
    selected_hedge_ratio = 1.0

print("\nInitializing CointegrationDollarNeutral Strategy...")

# Set strategy parameters
z_entry = 1.5  # Lowering the entry threshold from 2.0 to 1.5
z_exit = 0.0
lookback_window = 126  # Shortened to 6 months

strategy = CointegrationDollarNeutral(
    symbol_A=symbols[0],
    symbol_B=symbols[1],
    cash_symbol=cash_symbol,
    hedge_ratio=selected_hedge_ratio,
    lookback_window=lookback_window,
    z_entry=z_entry,
    z_exit=z_exit,
    book_size=0.5,
    tolerance=0.05
)

# Backtesting using ScheduleBasedEngine
schedule_name = 'weekly_friday' 
print(f"\nRunning backtest with ScheduleBasedEngine (Schedule: {schedule_name})...")

try:
    schedule = Schedule(schedule_name)
    
    engine = ScheduleBasedEngine(
        allocation=strategy,
        rebalance=schedule,
        initial_value=TOTAL_CAPITAL,
        fee_per_share=0.005
    )
    
    result = engine.run(
        symbols=all_symbols,
        start=start_date,
        end=end_date,
        prices_df=prices_data 
    )

    metrics = compute_metrics(result.equity_curve)
    
    print("\n" + "="*60)
    print("🏆 BACKTEST RESULTS SUMMARY")
    print("="*60)
    
    final_equity = result.equity_curve.iloc[-1]
    total_return = (final_equity / TOTAL_CAPITAL - 1) * 100
    
    print(f"Final Equity:   ${final_equity:,.2f}")
    print(f"Total Return:   {total_return:.2f}%")
    print(f"Sharpe Ratio:   {metrics.get('sharpe_ratio', 0):.3f}")
    print(f"Max Drawdown:   {metrics.get('max_drawdown', 0):.2%}")
    print(f"Annual Return:  {metrics.get('annualized_return', 0):.2%}")
    print(f"Annual Vol:     {metrics.get('annualized_volatility', 0):.2%}")
    print(f"Sortino Ratio:  {metrics.get('sortino_ratio', 0):.3f}")
    
    # Calculate the number of actual transactions.
    # Identify rebalance decisions with weights different from the previous one.
    trade_count = 0
    prev_weights = None
    for dec in result.rebalance_decisions:
        curr_weights = getattr(dec, 'target_weights', {})
        if prev_weights is not None and curr_weights != prev_weights:
            trade_count += 1
        prev_weights = curr_weights
        
    print(f"Number of Trade Events: {trade_count}")
    print("="*60)

except Exception as e:
    print(f"❌ Backtest failed: {e}")
    import traceback
    traceback.print_exc()

Loading data for ['KO', 'PEP', 'BIL'] from 2018-01-01 to 2024-12-31...
Loading bar data...


Loaded bar data: 0:00:02 

✅ Successfully fetched data for 3 symbols

Calculating Johansen Cointegration for KO/PEP...
⚠️ No strong cointegration found. Using default Hedge Ratio: 1.0

Initializing CointegrationDollarNeutral Strategy...

Running backtest with ScheduleBasedEngine (Schedule: weekly_friday)...

🏆 BACKTEST RESULTS SUMMARY
Final Equity:   $116,493.06
Total Return:   16.49%
Sharpe Ratio:   8.613
Max Drawdown:   0.20%
Annual Return:  0.00%
Annual Vol:     0.00%
Sortino Ratio:  30.699
Number of Trade Events: 0


In [ ]:
# Prepare Universe stocks for dynamic stock selection
stock_universe = ['AAPL', 'MSFT', 'NVDA', 'INTC', 'AMD', 'META', 'AMZN', 'GOOGL', 'TSLA', 'NFLX']
cash_symbol = 'BIL'
all_symbols = stock_universe + [cash_symbol]

print(f"Loading data for Universe ({len(stock_universe)} stocks) from {start_date} to {end_date}...")
try:
    prices_data = fetch_prices(all_symbols, start_date, end_date)
    print("✅ Successfully fetched data.")
except Exception as e:
    print(f"❌ Error fetching data: {e}")
    raise

In [ ]:
# Define dynamic stock selection long/short strategy
class DynamicMomentumDollarNeutral(DollarNeutral):
    """
    On each rebalancing day, review the returns over the past N days.
    Go long on the top_n best-performing stocks and short on the bottom_n worst-performing stocks.
    Maintain a neutral cash position.
    """
    def __init__(self, universe: list[str], cash_symbol: str, 
                 lookback_window: int = 21, top_n: int = 2, 
                 book_size: float = 0.5, tolerance: float = 0.05):
        
        self.universe = universe
        self.lookback_window = lookback_window
        self.top_n = top_n
        
        super().__init__(
            long_weights={universe[0]: 1.0},
            short_weights={universe[1]: 1.0},
            cash_symbol=cash_symbol,
            book_size=book_size,
            tolerance=tolerance
        )

    def get_symbols(self) -> list[str]:
        """
        Tell Engine which stock price data our strategy needs.
        Send back the entire Univers because dynamically selecting stocks.
        """
        return self.universe + [self.cash_symbol]

    def get_target_weights(self, date: pd.Timestamp, total_equity: float, 
                           positions_dollars: dict[str, float], 
                           prices_row: pd.Series, **context: Any) -> dict[str, float]:
        
        prices_history = context.get("prices_history")
        
        # 1. Check if the historical data is sufficient
        if prices_history is None or len(prices_history) < self.lookback_window:
            # Empty-handed when there is insufficient information
            self.long_weights = {self.universe[0]: 0.0}
            self.short_weights = {self.universe[1]: 0.0}
            self.long_book_size = 0.0
            self.short_book_size = 0.0
            return super().get_target_weights(date, total_equity, positions_dollars, prices_row, **context)

        # 2. Calculate the cumulative rate of return over the past lookback_window days
        # Extract the historical prices of Universe stocks
        hist_prices = prices_history[self.universe].tail(self.lookback_window)
        # Rate of return = (Today's price / Price N days ago) - 1
        returns = (hist_prices.iloc[-1].astype(float) / hist_prices.iloc[0].astype(float)) - 1
        
        # 3. Sort to identify winners and losers
        returns = returns.dropna()
        
        if len(returns) >= self.top_n * 2:
            sorted_returns = returns.sort_values(ascending=False)
            
            # Find the N stocks with the largest price changes
            winners = sorted_returns.head(self.top_n).index.tolist()
            losers = sorted_returns.tail(self.top_n).index.tolist()
            
            # 4. Distribute weights evenly
            # If 2 are selected, each will account for 0.5 (50%) of the Long Book.
            weight_per_stock = 1.0 / self.top_n
            
            self.long_weights = {sym: weight_per_stock for sym in winners}
            self.short_weights = {sym: weight_per_stock for sym in losers}
            
            # Restore normal fund size
            self.long_book_size = self.book_size
            self.short_book_size = self.book_size
            
            print(f"[{date.date()}] Long: {winners} | Short: {losers}")
            
        else:
            # Insufficient stock quantity, remain uninvested.
            self.long_book_size = 0.0
            self.short_book_size = 0.0

        # 5. Calling the parent category to handle tolerance and cash allocation
        return super().get_target_weights(date, total_equity, positions_dollars, prices_row, **context)

# 3. Perform backtesting (monthly portfolio rebalancing)
print("\nInitializing DynamicMomentumDollarNeutral Strategy...")

# 
strategy = DynamicMomentumDollarNeutral(
    universe=stock_universe,
    cash_symbol=cash_symbol,
    lookback_window=21,  # returns over the past 21 days
    top_n=2,             # top 2 long & bottom 2 short positions
    book_size=0.5,
    tolerance=0.05
)

# Momentum strategies typically involve monthly portfolio rebalancing.
schedule_name = 'month_end' 
print(f"\nRunning backtest with ScheduleBasedEngine (Schedule: {schedule_name})...")

try:
    schedule = Schedule(schedule_name)
    
    engine = ScheduleBasedEngine(
        allocation=strategy,
        rebalance=schedule,
        initial_value=TOTAL_CAPITAL,
        fee_per_share=0.005
    )
    
    result = engine.run(
        symbols=all_symbols,
        start=start_date,
        end=end_date,
        prices_df=prices_data 
    )
    
    metrics = compute_metrics(result.equity_curve)
    
    print("\n" + "="*60)
    print("🏆 DYNAMIC MOMENTUM STRATEGY RESULTS")
    print("="*60)
    
    final_equity = result.equity_curve.iloc[-1]
    total_return = (final_equity / TOTAL_CAPITAL - 1) * 100
    
    print(f"Final Equity:   ${final_equity:,.2f}")
    print(f"Total Return:   {total_return:.2f}%")
    print(f"Sharpe Ratio:   {metrics.get('sharpe_ratio', 0):.3f}")
    print(f"Max Drawdown:   {metrics.get('max_drawdown', 0):.2%}")
    print(f"Annual Return:  {metrics.get('annualized_return', 0):.2%}")
    
    trade_count = 0
    prev_weights = None
    for dec in result.rebalance_decisions:
        curr_weights = getattr(dec, 'target_weights', {})
        if prev_weights is not None and curr_weights != prev_weights:
            trade_count += 1
        prev_weights = curr_weights
        
    print(f"Number of Trade Events: {trade_count}")
    print("="*60)

except Exception as e:
    print(f"❌ Backtest failed: {e}")

Loading data for Universe (10 stocks) from 2018-01-01 to 2024-12-31...
Loading bar data...
Loaded bar data: 0:00:03 

✅ Successfully fetched data.

Initializing DynamicMomentumDollarNeutral Strategy...

Running backtest with ScheduleBasedEngine (Schedule: month_end)...
[2018-02-28] Long: ['AMZN', 'AAPL'] | Short: ['GOOGL', 'AMD']
[2018-03-29] Long: ['INTC', 'NFLX'] | Short: ['AMD', 'TSLA']
[2018-04-30] Long: ['AMD', 'TSLA'] | Short: ['NVDA', 'AAPL']
[2018-05-31] Long: ['AMD', 'NFLX'] | Short: ['AMZN', 'TSLA']
[2018-06-29] Long: ['TSLA', 'NFLX'] | Short: ['NVDA', 'INTC']
[2018-07-31] Long: ['AMD', 'GOOGL'] | Short: ['NFLX', 'TSLA']
[2018-08-31] Long: ['AMD', 'NVDA'] | Short: ['INTC', 'TSLA']
[2018-09-28] Long: ['AMD', 'NFLX'] | Short: ['GOOGL', 'INTC']
[2018-10-31] Long: ['TSLA', 'INTC'] | Short: ['NVDA', 'AMD']
[2018-11-30] Long: ['AMD', 'AMZN'] | Short: ['AAPL', 'NVDA']
[2018-12-31] Long: ['META', 'TSLA'] | Short: ['NVDA', 'AMD']
[2019-01-31] Long: ['NFLX', 'AMD'] | Short: ['INTC', 'T